In [ ]:
import os
DREAMS_DATA = os.environ.get("DREAMS_DATA", "/path/to/DREAMS")
MWOBS_ROOT = os.environ.get("MWOBS_ROOT", ".")
NEHOD_TORCH = os.environ.get("NEHOD_TORCH", "/path/to/nehod_torch")

import numpy as np
from scipy.stats import binned_statistic as bs
from scipy.stats import binned_statistic_2d as bs2d
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import binned_statistic
import DREAMS
import os
import corner

In [ ]:
def set_ticks(ax):
    ax.tick_params('both', which='minor', length=4, direction='in', bottom=True, top=True, left=True, right=True)
    ax.tick_params('both', which='major', length=8, direction='in', bottom=True, top=True, left=True, right=True)

    for axis in ['top','bottom','left','right']:
        ax.spines[axis].set_linewidth(2)

    return

def set_plot_params(nrows=1, ncols=1, figsize=None, **kwargs):

    plt.rc('font',**{'family':'STIXGeneral'})
    plt.rc('text', usetex=False)
    plt.rc('mathtext', fontset='stix')

    plt.rc('font', size=12)          # controls default text sizes
    plt.rc('axes', titlesize=16)     # fontsize of the axes title
    plt.rc('axes', labelsize=24)    # fontsize of the x and y labels
    plt.rc('xtick', labelsize=18)    # fontsize of the tick labels
    plt.rc('ytick', labelsize=18)    # fontsize of the tick labels
    plt.rc('legend', fontsize=16)

    plt.rc('lines', linewidth=3)

    if figsize is None:
        fig, ax = plt.subplots(nrows,ncols, figsize=(ncols*3 + (ncols)*3,nrows*5+(nrows-1)*3))
    else:
        fig, ax = plt.subplots(nrows,ncols, figsize=figsize, **kwargs)

    if type(ax)==type(np.zeros(1)):
        for a in ax.ravel():
            set_ticks(a)
    else:
        set_ticks(ax)

    return fig, ax

In [ ]:
gen_cond = np.load(MWOBS_ROOT + '/nehod/generated_samples/1000k_CDM_SB_gse_prob.npy')
# gen_cond = np.load(NEHOD_TORCH + '/example/generated_samples/100k_CDM_CV_gse_prob_cond.npy')
print(gen_cond.shape)

In [ ]:
gen_cond[0]

In [ ]:
mass = gen_cond[:,0]
om  = gen_cond[:,1]
s8  = gen_cond[:,2]
sn1 = gen_cond[:,3]
sn2 = gen_cond[:,4]
agn = gen_cond[:,5]


#f beta
mm_time = gen_cond[:,6]
gse_time = gen_cond[:,7]
gse_ratio = gen_cond[:,8]
ngse = gen_cond[:,9]
halo_f = gen_cond[:,10]
halo_beta = gen_cond[:,11]

In [ ]:
cut = np.ones(len(mass),dtype=bool)

# cut[(halo_f < 0) | (halo_f > 1)] = False
# cut[(halo_beta < -5) | (halo_beta > 5)] = False
# print(np.sum(cut))

# cut[(mm_time<0) | (mm_time>1.0)]=False
# cut[(gse_time<0) | (gse_time>1.0)]=False
# cut[(gse_ratio<1) | (gse_ratio > 50)] = False
print(np.sum(cut))

In [ ]:
for data in gen_cond.T:
    plt.figure()
    plt.hist(data[cut],bins=100)
    plt.show()

In [ ]:
for i,data in enumerate(gen_cond.T):
    print(i)
    plt.figure()
    plt.hist2d(gen_cond[:,12], data[cut], bins=100)
    plt.show()

In [ ]:
print(len(halo_f))
fcut = halo_f > 0.5
bcut = halo_beta > 0.5
ncut = ngse < 3
ra_cut = fcut & bcut #& ncut
print(np.sum(ra_cut))

gcut = (gse_time < 0.5) & (gse_time > 0.33)
tcut = (mm_time < 0.5)
mcut = gse_ratio < 5  # Release: paper criterion M_MW/M_gse < 5 (was < 10); gives the 11 simulated analogs
gse_cut = gcut & tcut & mcut & ra_cut
print(np.sum(gse_cut))
print(np.sum(tcut), np.sum(mcut), np.sum(fcut), np.sum(bcut), np.sum(ncut))

In [ ]:
fig,ax = set_plot_params(figsize=(5,5))
ax.hist2d(mm_time, gse_time, bins=150, norm=LogNorm())
ax.plot([0,1],[0,1], 'k--')
ax.set_xlabel("Major Merger")
ax.set_ylabel("GSE Merger")
ax.axis("equal")
ax.set_xlim(0,1)
ax.set_ylim(0,1)
plt.show()

In [ ]:
plt.hist(ngse)
plt.show()

In [ ]:
gen_cond[0]

In [ ]:
dreams_data = np.loadtxt(MWOBS_ROOT + "/nehod/data/gse_prob_cond.csv", skiprows=1, delimiter=',')

In [ ]:
dreams_data[0]

In [ ]:
names = [r'$t_{\rm mm}$', r'$t_{\rm gse}$', r'$M_{\rm gse} / M_{\rm MW}$', '$f$', r'$\beta$', 'log $\mathrm{M_*}$ [M$_\odot$]']
# param_names = [r'$\Omega_m$', r'$\sigma_8$', '$e_w$', r'$\kappa_w$', r'$\epsilon_{f,\mathrm{high}}$']
param_names = [r'$\Omega_{\rm m}$', r'$\sigma_8$', 'log $\widebar{e}_w$', r'log $\kappa_w$', r'log $\epsilon_{f,\mathrm{high}}$']
param = 3
for i,idx in enumerate([6,7,8,10,11,12]):
    fig, ax = set_plot_params()
    # plt.hist2d(gen_cond[:,3], gen_cond[:,idx], bins=100)
    bins = np.linspace(np.min(gen_cond[:,param]), np.max(gen_cond[:,param]), 21)
    bc = (bins[1:]+bins[:-1])/2

    avg = binned_statistic(dreams_data[:,param+1], dreams_data[:,idx+1], 'mean', bins=bins)[0]
    std = binned_statistic(dreams_data[:,param+1], dreams_data[:,idx+1], 'std', bins=bins)[0]
    plt.plot(bc, avg, '-', color='olivedrab', label='DREAMS Simulation')
    print(avg[0], avg[-1])
    plt.fill_between(bc, avg+std, avg-std, color='olivedrab', alpha=0.3)
    
    avg = binned_statistic(gen_cond[:,param], gen_cond[:,idx], 'mean', bins=bins)[0]
    std = binned_statistic(gen_cond[:,param], gen_cond[:,idx], 'std', bins=bins)[0]
    plt.plot(bc, avg, 'k-', label='Emulated MW-Mass Galaxies')
    print(avg[0], avg[-1])
    plt.fill_between(bc, avg+std, avg-std, color='k', alpha=0.3)
    
    plt.ylabel(names[i])
    plt.xlabel(f"{param_names[param-1]}")
    
    plt.legend(frameon=False)
    # plt.ylim(9.7,11.1)
    
    plt.show()
    # fig.savefig(f"plots/1D_distributions/{param}_vs_{idx}.pdf", bbox_inches='tight')

In [ ]:
list(zip(np.arange(len(gen_cond[0])),np.min(gen_cond,axis=0),np.max(gen_cond,axis=0) )) 

In [ ]:
names = [r'$t_{\rm mm}$', 'tgse', r'$M_{\rm MW}/M_{\rm gse}$', '$f$', r'$\beta$'] # [6,7,8,10,11]
param_idx = {6:0, 7:1, 8:2, 10:3, 11:4}
param_names = ['om', 's8', 'ew', 'kw', 'ef'] # [1,2,3,4,5]
mw_names = ['log SFR [M$_\odot$ yr$^{-1}$]', '$\mathrm{R_{disk}}$ [kpc]', 'log $\mathrm{M_{*,halo}}$ [M$_\odot$]', '$r_s$ [kpc]', r'$\alpha_{\rm in}$', r'$q_{\rm in}$'] #[16,25,27,28,29,31]
labels = [r'$t_{\rm mm}$', r'$t_{\rm gse}$', r'$M_{\rm gse} / M_{\rm MW}$', r'N$_{\rm gse}$', '$f$', r'$\beta$']
labels += ['log $\mathrm{M_*}$ [M$_\odot$]', 'log $\mathrm{M_{*,sat}}$ [M$_\odot$]', 'log $\mathrm{M_{bary}}$ [M$_\odot$]', 'log SFR [M$_\odot$ yr$^{-1}$]']
labels += ['$\mathrm{R_{eff}}$ [kpc]', 'log $\mathrm{M_{TOT,bulge}}$ [M$_\odot$]', 'log $\mathrm{M_{*,bulge}}$ [M$_\odot$]', 'log $\mathrm{M_{BH}}$ [M$_\odot$]', '$\sigma_\mathrm{*,bulge}$ [km s$^{-1}$]']
labels += ['V$_\mathrm{circ}$  [km s$^{-1}$]', 'log $\mathrm{M_{*,disk}}$  [M$_\odot$]', '$\mathrm{R_{disk}}$ [kpc]', '$z_\mathrm{disk}$ [kpc]']
labels += ['log $\mathrm{M_{*,halo}}$ [M$_\odot$]', '$r_s$ [kpc]', r'$\alpha_\mathrm{in}$', r'$\alpha_\mathrm{out}$', "q$_\mathrm{in}$", "q$_\mathrm{out}$"]
ma = [1,1,50,1,1]
param = 11
for i,idx in enumerate([16,25,27,28,29,31]):
    fig, ax = set_plot_params()
    # plt.hist2d(gen_cond[:,3], gen_cond[:,idx], bins=100)
    bins = np.linspace(0.1, ma[param_idx[param]], 21)
    bc = (bins[1:]+bins[:-1])/2
    
    avg = binned_statistic(gen_cond[:,param], gen_cond[:,idx], 'mean', bins=bins)[0]
    std = binned_statistic(gen_cond[:,param], gen_cond[:,idx], 'std', bins=bins)[0]
    plt.plot(bc, avg, 'k-')
    print(avg[0], avg[-1])
    plt.fill_between(bc, avg+std, avg-std, color='k', alpha=0.3)

    avg = binned_statistic(dreams_data[:,param+1], dreams_data[:,idx+1], 'mean', bins=bins)[0]
    std = binned_statistic(dreams_data[:,param+1], dreams_data[:,idx+1], 'std', bins=bins)[0]
    plt.plot(bc, avg, '-', color='olivedrab')
    print(avg[0], avg[-1])
    plt.fill_between(bc, avg+std, avg-std, color='olivedrab', alpha=0.3)

    # plt.ylim(0.3,0.8)
    
    plt.xlabel(names[param_idx[param]])
    plt.ylabel(f"{mw_names[i]}")
    plt.show()
    fig.savefig(f"plots/1D_distributions/{param_idx[param]}_vs_{i}.pdf", bbox_inches='tight')

In [ ]:
mi = [0.274, 0.780, np.log10(0.9), np.log10(3.7), np.log10(0.025)]
ma = [0.354, 0.888, np.log10(14.4), np.log10(14.8), np.log10(0.4)]
names = ['$\Omega_m$', '$\sigma_8$', '$\widebar{e}_w$', '$k_w$', '$\epsilon_{f,high}$']
fig, ax = set_plot_params()
for i,param in enumerate(gen_cond.T[1:6]):
    cnt, be, bn = bs((param[gse_cut]-mi[i])/(ma[i]-mi[i]), mm_time[gse_cut], 'count', bins=20)
    tot, be, bn = bs((param[cut]-mi[i])/(ma[i]-mi[i]), mm_time[cut], 'count', bins=20)
    bc = (be[1:]+be[:-1])/2
    plt.plot(bc, cnt/tot, '-', label=names[i])

plt.legend(frameon=False, ncols=3)
plt.xlabel("Normalized Simulation Parameter")
plt.ylabel("Fraction of GSE Systems")
# plt.ylim((0,0.06))
plt.show()

In [ ]:
mi = [0.274, 0.780, np.log10(0.9), np.log10(3.7), np.log10(0.025)]
ma = [0.354, 0.888, np.log10(14.4), np.log10(14.8), np.log10(0.4)]
names = ['$\Omega_m$', '$\sigma_8$', '$\widebar{e}_w$', '$k_w$', '$\epsilon_{f,high}$']
colors = ['k', 'r', 'm', 'b', 'green']
colors = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE']
styles = ['--', '--', '-', '-', ':']
# colors = ['#0072B2', '#E69F00', '#F0E442', '#009E73', '#56B4E9']
# colors = ['#1B9E77', '#D95F02', '#7570B3', '#E7298A', '#66A61E']
all_params = ['Om', 's8', 'SN1', 'SN2', 'AGN']
# all_params = ['Om', 's8','SN1']

all_data = dict()

fig, ax = set_plot_params()

all_gse = []
for i,param in enumerate(all_params):

    # if i != 3:
    #     continue
    
    for j in range(10):
        data = np.load(f"nehod/generated_samples/100k_CDM_{param}_{j}_gse_prob.npy")
        
        fcut = data[:,10] > 0.5
        bcut = data[:,11] > 0.5
        gse_cut = fcut & bcut #tcut & mcut & fcut & bcut & cut

        cnt, be, bn = bs((data[:,i+1][gse_cut]-mi[i])/(ma[i]-mi[i]), data[:,7][gse_cut], 'count', bins=6)
        tot, be, bn = bs((data[:,i+1]-mi[i])/(ma[i]-mi[i]), data[:,7], 'count', bins=6)
        bc = (be[1:]+be[:-1])/2

        all_gse.append(np.sum(gse_cut) / len(gse_cut))
        
        if j == 0:
            all_data[param] = [cnt/tot]
            # plt.plot(bc, cnt/tot, '-', label=names[i], color=colors[i], alpha=0.5)
        else:
            all_data[param].append(cnt/tot)
            # plt.plot(bc, cnt/tot, '-', color=colors[i], alpha=0.5)

for i,param in enumerate(all_params):
    # if i != 3:
    #     continue
    avg = np.mean(all_data[param],axis=0)
    std = np.std(all_data[param],axis=0)
    plt.plot(bc, avg, label=names[i], color=colors[i], linestyle=styles[i])
    plt.fill_between(bc, avg+std, avg-std, color=colors[i], alpha=0.3)

avg = np.average(all_gse)
std = np.std(all_gse)
print(avg, std)

plt.legend(frameon=False, ncols=3)
plt.xlabel("Normalized Simulation Parameter")
plt.ylabel("Fraction of Systems")
# plt.ylim((0,0.35))
plt.text(0.1,0.31,'RA Merger', fontsize=20)
# plt.yscale("log")
plt.show()
# fig.savefig("plots/ra_merger_params.pdf", bbox_inches='tight')

In [ ]:
bc, np.average(all_data['SN2'],axis=0), np.std(all_data['SN2'],axis=0) 

In [ ]:
mi = [0.274, 0.780, np.log10(0.9), np.log10(3.7), np.log10(0.025)]
ma = [0.354, 0.888, np.log10(14.4), np.log10(14.8), np.log10(0.4)]
names = ['$\Omega_m$', '$\sigma_8$', '$\widebar{e}_w$', '$k_w$', '$\epsilon_{f,high}$']
colors = ['k', 'r', 'm', 'b', 'green']
all_params = ['Om', 's8', 'SN1', 'SN2', 'AGN']
colors = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE']
styles = ['--', '--', '-', '-', ':']

all_data = dict()

fig, ax = set_plot_params()

all_gse = []
for i,param in enumerate(all_params):

    # if i != 3:
    #     continue
    
    for j in range(10):
        data = np.load(f"nehod/generated_samples/100k_CDM_{param}_{j}_gse_prob.npy")
        
        tcut = data[:,6] < 0.5
        gse_tcut = (data[:,7] < 0.5) & (data[:,7] > 0.33)
        mcut = data[:,8] < 5 
        fcut = data[:,10] > 0.5
        bcut = data[:,11] > 0.5
        gse_cut = fcut & bcut & gse_tcut #& tcut & gse_tcut & mcut 

        cnt, be, bn = bs((data[:,i+1][gse_cut]-mi[i])/(ma[i]-mi[i]), data[:,7][gse_cut], 'count', bins=6)
        tot, be, bn = bs((data[:,i+1]-mi[i])/(ma[i]-mi[i]), data[:,7], 'count', bins=6)
        bc = (be[1:]+be[:-1])/2

        all_gse.append(np.sum(gse_cut) / len(gse_cut))
        
        if j == 0:
            all_data[param] = [cnt/tot]
            # plt.plot(bc, cnt/tot, '-', label=names[i], color=colors[i], alpha=0.5)
        else:
            all_data[param].append(cnt/tot)
            # plt.plot(bc, cnt/tot, '-', color=colors[i], alpha=0.5)

for i,param in enumerate(all_params):
    # if i != 3:
    #     continue
    avg = np.mean(all_data[param],axis=0)
    std = np.std(all_data[param],axis=0)
    print(np.mean(avg))
    plt.plot(bc, avg, label=names[i], color=colors[i], linestyle=styles[i])
    plt.fill_between(bc, avg+std, avg-std, color=colors[i], alpha=0.3)

avg = np.average(all_gse)
std = np.std(all_gse)
print(avg, std)
# plt.plot(bc, [avg]*len(bc), 'k--')
# plt.fill_between(bc, [std+avg]*len(bc), [avg-std]*len(bc), color='k', alpha=0.3)

# plt.legend(frameon=False, ncols=3)
plt.xlabel("Normalized Simulation Parameter")
plt.ylabel("Fraction of Systems")
# plt.ylim((0,0.03))
# plt.text(0.1,0.026,'GSE Merger', fontsize=20)
# plt.yscale("log")
plt.legend(loc='lower right', frameon=False)
plt.show()
# fig.savefig("plots/gse_merger_params.pdf", bbox_inches='tight')

In [ ]:
bc, np.average([all_data[param] for param in all_data]), np.std([all_data[param] for param in all_data]) 

In [ ]:
def get_bin_idx(point, npoints=30):

    sn1 = np.linspace(np.log10(0.9), np.log10(14.4), npoints)
    sn2 = np.linspace(np.log10(3.7), np.log10(14.8), npoints)
    agn = np.linspace(np.log10(0.025), np.log10(0.4), npoints)

    bins = [sn1, sn2, agn]
    all_idx = []
    for i in range(len(bins)):
        idx = np.argmin(np.abs((bins[i] - point[i])))
        all_idx.append(idx)

    return np.array(all_idx)

In [ ]:
# galaxy_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_galaxy_dylan_cond.npy')
# bulge_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_bulge_dylan_cond.npy')
# disk_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_disk_dylan_cond.npy')
# halo_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_halo_dylan_cond.npy')
# galaxy_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_galaxy_dylan_cond.npy')
# bulge_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_bulge2_dylan_cond.npy')
# disk_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_disk_dylan_cond.npy')
# halo_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_halo_dylan_cond.npy')

In [ ]:
galaxy_data = np.load(MWOBS_ROOT + '/nehod/generated_samples/1000k_CDM_SB_gse_prob.npy')
# galaxy_data = np.load(MWOBS_ROOT + '/nehod/generated_samples/1000k_CDM_CV_gse_prob.npy')
# galaxy_data = np.load(MWOBS_ROOT + '/nehod/generated_samples/1000k_CDM_low_gse_prob.npy')

In [ ]:
galaxy_data[12]

In [ ]:
# binned_weights = np.load("binned_weights.npy")
binned_weights = np.load("nehod_weights/arya_weights_binned.npy")

In [ ]:
names = ['halo_tot_smass','halo_sat_smass', 'halo_bmass', 'halo_bmass_frac', 'halo_sfr']
ngal = len(names)
names += ['bulge_reff', 'bulge_mass', 'bulge_smass', 'bulge_bhmass', 'bulge_dm_frac', 'bulge_vdisp']
nbulge = len(names) - ngal
names += ['disk_vcirc', 'disk_smass', 'disk_size', 'disk_height']
ndisk = len(names) - nbulge - ngal
names += ['halo_central_smass',  'halo_break', 'halo_slope_in', 'halo_slope_out', 'halo_q_in', 'halo_q_out']
nhalo = len(names) - nbulge - ndisk - ngal

In [ ]:
# names = ['bulge_reff', 'bulge_mass', 'bulge_smass', 'bulge_bhmass', 'bulge_dm_frac', 'bulge_vdisp']

In [ ]:
full_data = []
full_weights = []
for all_data in [galaxy_data]:

    gse_cut = np.zeros(len(all_data),dtype=bool)

    shuffle_idx = np.arange(len(all_data))
    np.random.shuffle(shuffle_idx)
    all_data = all_data[shuffle_idx]
    print(all_data.shape)

    end = 10000 
    # end = 1000000
    # end = 500000
    while np.sum(gse_cut) < 13613:
        end += 10000
        # end += 1
        tcut = all_data[:end,6] < 0.5
        gse_tcut = (all_data[:end,7] < 0.5) & (all_data[:end,7] > 0.33)
        mcut = all_data[:end,8] < 5
        fcut = all_data[:end,10] > 0.5
        bcut = all_data[:end,11] > 0.5
        # gse_cut = tcut & gse_tcut & fcut & bcut & mcut
        gse_cut = np.ones(end,dtype=bool)
    print(len(gse_cut), np.sum(gse_cut))

    for data in all_data.T:
        full_data.append(data[:end][gse_cut])
    
weights = []
for k in np.arange(end)[gse_cut]:
    if k % (np.sum(gse_cut) // 5) == 0:
        print(len(weights) // (np.sum(gse_cut) // 5))
    idx = get_bin_idx(all_data[k,3:6])
    w = binned_weights[idx[0]][idx[1]][idx[2]]
    weights.append(w)
full_weights = np.array(weights)
        

In [ ]:
full_data = np.array(full_data).T
full_weights = np.array(full_weights).T

In [ ]:
full_data.shape, full_weights.shape

In [ ]:
em_good = np.arange(33)[(np.arange(33)>11) & (np.arange(33)!=15) & (np.arange(33)!=21)]

In [ ]:
full_data[0,em_good]

In [ ]:
# np.save(MWOBS_ROOT + "/gse_data", full_data[:,em_good])
# np.save(MWOBS_ROOT + "/gse_weights", full_weights[:,em_good])
# np.save(MWOBS_ROOT + "/full_data", full_data[:,em_good])
# np.save(MWOBS_ROOT + "/full_weights", full_weights[:,em_good])
# np.save(MWOBS_ROOT + "/TNG_data", full_data[:,em_good])
# np.save(MWOBS_ROOT + "/TNG_weights", full_weights)

In [ ]:
import corner

In [ ]:
# dreams_data = np.loadtxt(NEHOD_TORCH + "/example/data/gse_prob_bulge_cond.csv", skiprows=1, delimiter=',')
dreams_data = np.loadtxt(MWOBS_ROOT + "/nehod/data/gse_prob_cond.csv", skiprows=1, delimiter=',')

In [ ]:
dreams_data.shape

In [ ]:
print(np.min(full_data[:,0:6],axis=0), np.max(full_data[:,0:6],axis=0))
print(np.min(dreams_data[:,1:7],axis=0), np.max(dreams_data[:,1:7],axis=0))

In [ ]:
dreams_weights = []
for k in np.arange(len(dreams_data)):
    idx = get_bin_idx(dreams_data[k,4:7])
    w = binned_weights[idx[0]][idx[1]][idx[2]]
    dreams_weights.append(w)
dreams_weights = np.array(dreams_weights)

In [ ]:
# em_good = np.arange(33)[(np.arange(33)>11) & (np.arange(33)!=15) & (np.arange(33)!=21)]
# dr_good = np.arange(34)[(np.arange(34)>12) & (np.arange(34)!=16) & (np.arange(34)!=22)]
em_good = np.arange(33)[(np.arange(33)>0) & (np.arange(33)!=15) & (np.arange(33)!=21)]
dr_good = np.arange(34)[(np.arange(34)>1) & (np.arange(34)!=16) & (np.arange(34)!=22)]

In [ ]:
print(full_data.shape)
print(dreams_data.shape)

In [ ]:
# emulated_data = full_data[:,em_good][:,[9,13,17,19,20,21,23,24]] 
# true_data = dreams_data[:,dr_good][:,[9,13,17,19,20,21,23,24]] 
# emulated_data = full_data[:,em_good][:,[0,1,2,3,4,5]]
# true_data = dreams_data[:,dr_good][:,[0,1,2,3,4,5]] 

emulated_data = full_data[:,em_good]
true_data = dreams_data[:,dr_good]

In [ ]:
emulated_data.shape, true_data.shape

In [ ]:
full_weights.shape

In [ ]:
np.min(true_data,axis=0),np.max(true_data,axis=0)

In [ ]:
tcut = true_data[:end,0+5] < 0.5
gse_tcut = (true_data[:end,1+5] < 0.5) & (true_data[:end,1+5] > 0.33)
mcut = true_data[:end,2+5] < 5
fcut = true_data[:end,4+5] > 0.5
bcut = true_data[:end,5+5] > 0.5
gse_cut = tcut & gse_tcut & fcut & bcut & mcut

In [ ]:
np.sum(gse_cut), np.sum(tcut), np.sum(gse_tcut), np.sum(mcut), np.sum(fcut), np.sum(bcut), len(mcut)

In [ ]:
np.sum(fcut & gse_tcut & mcut & tcut)

In [ ]:
repeats = dreams_weights*100/np.mean(dreams_weights)
# wdata = np.repeat(true_data, repeats.astype(int), axis=0)
wdata = np.repeat(true_data[gse_cut], repeats[gse_cut].astype(int), axis=0)
true_data = wdata
# wdata.shape

In [ ]:
bins= np.logspace(-3,0)
# plt.hist(dreams_weights, bins=bins, density=True)
# plt.hist(full_weights, bins=bins, density=True)
# plt.plot(true_data[:,2], dreams_weights, '.')
# plt.hist2d(full_data[:,3], full_data[:,4], weights=full_weights, bins=100)
# plt.hist2d(true_data[:,2], true_data[:,3], weights=dreams_weights, bins=100)
# plt.xscale("log")
plt.show()

In [ ]:
repeats = full_weights*10/np.mean(full_weights)
wdata = np.repeat(emulated_data, repeats.astype(int), axis=0)
emulated_data = wdata
wdata.shape

In [ ]:
dreams_data_all = true_data
dreams_data_gse = true_data[gse_cut]

In [ ]:
dreams_data_all.shape #, dreams_data_gse.shape

In [ ]:
emulated_data_dreams = emulated_data
# emulated_data_gse = emulated_data
# emulated_data_tng = emulated_data
# emulated_data_tng_gse = emulated_data

In [ ]:
emulated_data.shape

In [ ]:
true_data.shape

In [ ]:
# emulated_data.shape, emulated_data_gse.shape

In [ ]:
emulated_data_dreams.shape

In [ ]:
# for i in range(10):
#     plt.figure()
#     plt.plot(true_data[:,0],true_data[:,i], '.')
#     plt.ylabel(labels[i])
#     plt.show()

In [ ]:
# Define the labels for each parameter. Using LaTeX for nice formatting

labels = [r'$\Omega_{\rm m}$', r'$\sigma_8$', 'log $\widebar{e}_w$', r'log $\kappa_w$', r'log $\epsilon_{f,\mathrm{high}}$']
labels += [r'$t_{\rm mm}$', r'$t_{\rm gse}$', r'$M_{\rm gse} / M_{\rm MW}$', r'N$_{\rm gse}$', '$f$', r'$\beta$']
labels += ['log $\mathrm{M_*}$ [M$_\odot$]', 'log $\mathrm{M_{*,sat}}$ [M$_\odot$]', 'log $\mathrm{M_{bary}}$ [M$_\odot$]', 'log SFR [M$_\odot$ yr$^{-1}$]']
labels += ['$\mathrm{R_{eff}}$ [kpc]', 'log $\mathrm{M_{TOT,bulge}}$ [M$_\odot$]', 'log $\mathrm{M_{*,bulge}}$ [M$_\odot$]', 'log $\mathrm{M_{BH}}$ [M$_\odot$]', '$\sigma_\mathrm{*,bulge}$ [km s$^{-1}$]']
labels += ['V$_\mathrm{circ}$  [km s$^{-1}$]', 'log $\mathrm{M_{*,disk}}$  [M$_\odot$]', '$\mathrm{R_{disk}}$ [kpc]', '$z_\mathrm{disk}$ [kpc]']
labels += ['log $\mathrm{M_{*,halo}}$ [M$_\odot$]', '$r_s$ [kpc]', r'$\alpha_\mathrm{in}$', r'$\alpha_\mathrm{out}$', "q$_\mathrm{in}$", "q$_\mathrm{out}$"]

# Release: this cell produces Figs A1, A2, B3 and B4 depending on which datasets are plotted:
#   A1: idx = [2,5,6,7,8,9,10] with true_data (green) + emulated_data_dreams (black)
#   A2: idx below with true_data (green) + emulated_data_dreams (black)
#   B3: idx below with emulated_data_dreams (black) + emulated_data_gse (red)
#   B4: idx below with emulated_data_dreams (black) + emulated_data_tng (blue) + emulated_data_tng_gse (purple)  [default]
# Panels (paper): SFR, M_BH, R_disk, M*,halo, r_s, alpha_in, q_in, q_out.
idx = [14,18,22,24,25,26,28,29]
# idx = [15,18,19,20,22,26,28,29]  # (research-code override: Reff, M_BH, sigma, Vcirc, Rdisk, alpha_in, q_in, q_out)
# idx = [2,5,6,7,8,9,10]

# labels = np.array(labels)[[6,9,13,17,18,19,23]]
# labels = np.array(labels)[[9,13,17,19,20,21,23,24]]
# labels = np.array(labels)[[0,1,2,3,4,5]]
labels = np.array(labels)[idx]

# Define plot styles for the two datasets
# DREAMS Simulations (Truth) - Black
truth_color = 'olivedrab'
truth_plot_kwargs = dict(
    color=truth_color,
    # FIX: Replaced 'fillStyle':'none' with 'histtype':'step'
    hist_kwargs={'histtype': 'step', 'lw': 2, 'density': True},
    contour_kwargs={'linewidths': 2}
)

emulator_color = 'k'
emulator_plot_kwargs = dict(
    color=emulator_color,
    # FIX: Replaced 'fillStyle':'none' with 'histtype':'step'
    hist_kwargs={'histtype': 'step', 'lw': 2, 'density': True},
    contour_kwargs={'linewidths': 2}
)

# NeHOD Emulator (Model) - Red
emulator_gse_color = '#d62728' # A nice red color
emulator_gse_kwargs = dict(
    color=emulator_gse_color,
    hist_kwargs={'alpha': 0.7, 'lw': 2, 'density': True},
    contour_kwargs={'linewidths': 2}
)

# NeHOD Emulator (Model) - Red
emulator_tng_color = 'royalblue' 
emulator_tng_kwargs = dict(
    color=emulator_tng_color,
    hist_kwargs={'alpha': 0.7, 'lw': 2, 'density': True},
    contour_kwargs={'linewidths': 2}
)

# NeHOD Emulator (Model) - Red
emulator_tng_gse_color = 'm' 
emulator_tng_gse_kwargs = dict(
    color=emulator_tng_gse_color,
    hist_kwargs={'alpha': 0.7, 'lw': 2, 'density': True},
    contour_kwargs={'linewidths': 2}
)

# --- 3. Create the Corner Plot ---

# combined_data = np.vstack([true_data, emulated_data])
# plot_range = [(np.quantile(col,0.01), np.quantile(col,0.99)) for col in true_data.T]
# 12.29995948  0.35399288  0.8879989   1.1582861   1.170201   -0.39810918
plot_range = [(0.27, 0.36), (0.78, 0.888), (-0.045, 1.16), (0.568, 1.18), (-1.6, -0.4)]
plot_range += [(0,1),(0,1),(0,49),(0,3.2),(0,1),(-1,1)]
plot_range += [(9.7,11.2),(7,11),(10.6,11.4),(-4.5,1.5),(0.2,1),(9.4,10.8),(8.8,10.7),(6.5,9),(65,350),(100,300),(8.6,11.2),(1,8),(0,1.6),(8.4,10.2),(11,60),(-3.4,-1.2),(-5.5,-2.5),(0.4,0.95),(0.25,0.95)] 
# plot_range = np.array(plot_range)[[6,9,13,17,18,19,23],:]
# plot_range = np.array(plot_range)[[9,13,17,19,20,21,23,24],:]
# plot_range = np.array(plot_range)[[0,1,2,3,4,5],:]
plot_range = np.array(plot_range)[idx, :]
print(plot_range)

# Create the figure object. The corner.corner function returns the figure.
# We plot the "true" data first as a reference.
ncols = true_data[:,idx].shape[1]
fig, ax = set_plot_params(ncols,ncols,figsize=(ncols*2,ncols*2))

# corner.corner(
#     true_data[:,idx],
#     fig=fig,
#     labels=labels,
#     plot_datapoints=False,
#     plot_density=False,
#     plot_contours=True,
#     range=plot_range,
#     bins=10,
#     levels=[0.393, 0.864],
#     **truth_plot_kwargs
# )

corner.corner(
    emulated_data_dreams[:,idx], 
    fig=fig,
    labels=labels,
    plot_datapoints=False,
    plot_density=False,
    plot_contours=True,
    range=plot_range,
    bins=20,
    levels=[0.393, 0.864],
    **emulator_plot_kwargs
)

# corner.corner(
#     emulated_data_gse[:,idx], 
#     fig=fig,
#     labels=labels,
#     plot_datapoints=False,
#     plot_density=False,
#     plot_contours=True,
#     range=plot_range,
#     bins=20,
#     levels=[0.393, 0.864],
#     **emulator_gse_kwargs
# )

corner.corner(
    emulated_data_tng[:,idx], 
    fig=fig,
    labels=labels,
    plot_datapoints=False,
    plot_density=False,
    plot_contours=True,
    range=plot_range,
    bins=20,
    levels=[0.393, 0.864],
    **emulator_tng_kwargs
)

corner.corner(
    emulated_data_tng_gse[:,idx],
    fig=fig,  # Pass the existing figure to overlay the plots
    plot_datapoints=False,
    plot_density=False,
    plot_contours=True,
    range=plot_range,
    bins=20,
    levels=[0.393, 0.864],
    **emulator_tng_gse_kwargs
)

# --- 4. Customization and Legend ---

# Adjust plot aesthetics for publication quality
axes = np.array(fig.get_axes())
for ax in axes:
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.xaxis.label.set_size(14)
    ax.yaxis.label.set_size(14)

# Add a custom legend to the plot
# We create proxy artists to represent the datasets in the legend
# A simple line is a good representation for both histograms and contours
legend_elements = [
    # plt.Line2D([0], [0], color=truth_color, lw=2, label='DREAMS Simulation'), # label='DREAMS Simulations (True)'),
    plt.Line2D([0], [0], color=emulator_color, lw=2, label='Emulated MW-Mass Galaxies (Varied Physics)'), # label='DREAMS Simulations (True)'),
    # plt.Line2D([0], [0], color=emulator_gse_color, lw=2, label='Emulated GSE Analogs (Varied Physics)'), # label='NeHOD Emulator (Model)')
    plt.Line2D([0], [0], color=emulator_tng_color, lw=2, label='Emulated All MW-Mass Galaxies (TNG Physics)'), # label='DREAMS Simulations (True)'),
    plt.Line2D([0], [0], color=emulator_tng_gse_color, lw=2, label='Emulated GSE Analogs (TNG Physics)') # label='NeHOD Emulator (Model)')
]

# Place the legend in a suitable position (e.g., top right)
# The coordinates are in figure space (0,0 is bottom left, 1,1 is top right)
fig.legend(handles=legend_elements, fontsize=18, loc='upper right', bbox_to_anchor=(0.55, 0.935), frameon=False)
# fig.legend(handles=legend_elements, fontsize=18, loc='upper right', bbox_to_anchor=(0.70, 0.935), frameon=False)

# Add a title to the entire figure
# fig.suptitle("Emulator Validation: DREAMS vs. NeHOD", fontsize=20)
plt.subplots_adjust(top=0.92) # Adjust top margin to make space for the title

# --- 5. Show the Plot ---
plt.show()
fig.savefig("plots/validation_GSE.pdf", bbox_inches='tight')

In [ ]:
# fig.savefig("plots/validation_GSE_selection.pdf", bbox_inches='tight')

In [ ]:
fig, ax = set_plot_params(2,3,figsize=(24,12))
lw = 4
#[15,18,19,20,22,26,28,29]

# labels += ['$\mathrm{R_{eff}}$ [kpc]', 'log $\mathrm{M_{TOT,bulge}}$ [M$_\odot$]', 'log $\mathrm{M_{*,bulge}}$ [M$_\odot$]', 'log $\mathrm{M_{BH}}$ [M$_\odot$]', '$\sigma_\mathrm{*,bulge}$ [km s$^{-1}$]']
# labels += ['V$_\mathrm{circ}$  [km s$^{-1}$]', 'log $\mathrm{M_{*,disk}}$  [M$_\odot$]', '$\mathrm{R_{disk}}$ [kpc]', '$z_\mathrm{disk}$ [kpc]']
# labels += ['log $\mathrm{M_{*,halo}}$ [M$_\odot$]', '$r_s$ [kpc]', r'$\alpha_\mathrm{in}$', r'$\alpha_\mathrm{out}$', "q$_\mathrm{in}$", "q$_\mathrm{out}$"]


bins = np.linspace(6,9,30)
ax[0,0].hist(emulated_data_dreams[:,18], bins=bins, histtype='step', color='royalblue', label='Varied Physics', linewidth=lw)
ax[0,0].hist(emulated_data_tng[:,18], bins=bins, histtype='step', color='m', label='Fixed Physics', linewidth=lw)
ax[0,0].set_xlabel('log $\mathrm{M_{BH}}$ [M$_\odot$]')
ax[0,0].legend(frameon=False, loc='upper right', fontsize=18)
ax[0,0].set_ylim((0,40000))

bins = np.linspace(100,350,30)
ax[0,1].hist(emulated_data_dreams[:,19], bins=bins, histtype='step', color='royalblue', linewidth=lw)
ax[0,1].hist(emulated_data_tng[:,19], bins=bins, histtype='step', color='m', linewidth=lw)
ax[0,1].set_xlabel('$\sigma_\mathrm{*,bulge}$ [km s$^{-1}$]')
ax[0,1].set_ylim((0,20000))

bins = np.linspace(-4.5,1.0,30)
ax[0,2].hist(emulated_data_dreams[:,14], bins=bins, histtype='step', color='royalblue', linewidth=lw)
ax[0,2].hist(emulated_data_tng[:,14], bins=bins, histtype='step', color='m', linewidth=lw)
ax[0,2].set_xlabel("log SFR [M$_\odot$ yr$^{-1}$")
ax[0,2].set_ylim((0,40000))

bins = np.linspace(100,300,30)
ax[1,0].hist(emulated_data_dreams[:,20], bins=bins, histtype='step', color='royalblue', linewidth=lw)
ax[1,0].hist(emulated_data_tng[:,20], bins=bins, histtype='step', color='m', linewidth=lw)
ax[1,0].set_xlabel('V$_\mathrm{circ}$  [km s$^{-1}$]')
ax[1,0].set_ylim((0,15000))

bins = np.linspace(1.4,7.5,30)
ax[1,1].hist(emulated_data_dreams[:,22], bins=bins, histtype='step', color='royalblue', linewidth=lw)
ax[1,1].hist(emulated_data_tng[:,22], bins=bins, histtype='step', color='m', linewidth=lw)
ax[1,1].set_xlabel('$\mathrm{R_{disk}}$ [kpc]')
ax[1,1].set_ylim((0,10000))

bins = np.linspace(-3.5,-1.0,30)
ax[1,2].hist(emulated_data_dreams[:,26], bins=bins, histtype='step', color='royalblue', linewidth=lw)
ax[1,2].hist(emulated_data_tng[:,26], bins=bins, histtype='step', color='m', linewidth=lw)
ax[1,2].set_xlabel(r'$\alpha_\mathrm{in}$')
ax[1,2].set_ylim((0,18000))

for i,axi in enumerate(ax.ravel()):
    axi.set_ylabel("Emulated Samples per Bin")
        

plt.show()
fig.savefig("plots/tng_comparison.png", bbox_inches='tight', dpi=100)

In [ ]:
fig, ax = set_plot_params(2,3,figsize=(24,12))
lw = 4

bins = np.linspace(-4.5,1.5,30)
ax[0,0].hist(emulated_data_dreams[:,14], bins=bins, histtype='step', color='k', label='Emulated MW-Mass Galaxies', linewidth=lw)
ax[0,0].hist(emulated_data_gse[:,14], bins=bins, histtype='step', color='r', label='Emulated GSE Analogs', linewidth=lw)
ax[0,0].set_xlabel('log SFR [M$_\odot$ yr$^{-1}$]')
ax[0,0].legend(frameon=False, loc='upper right', fontsize=18)
ax[0,0].set_ylim((0,30000))

bins = np.linspace(1,10,30)
ax[0,1].hist(emulated_data_dreams[:,22], bins=bins, histtype='step', color='k', linewidth=lw)
ax[0,1].hist(emulated_data_gse[:,22], bins=bins, histtype='step', color='r', linewidth=lw)
ax[0,1].set_xlabel('$\mathrm{R_{disk}}$ [kpc]')
ax[0,1].set_ylim((0,10000))

bins = np.linspace(8.4,10.2,30)
ax[0,2].hist(emulated_data_dreams[:,24], bins=bins, histtype='step', color='k', linewidth=lw)
ax[0,2].hist(emulated_data_gse[:,24], bins=bins, histtype='step', color='r', linewidth=lw)
ax[0,2].set_xlabel('log $\mathrm{M_{*,halo}}$ [M$_\odot$]')
ax[0,2].set_ylim((0,10000))

bins = np.linspace(11,65,30)
ax[1,0].hist(emulated_data_dreams[:,25], bins=bins, histtype='step', color='k', linewidth=lw)
ax[1,0].hist(emulated_data_gse[:,25], bins=bins, histtype='step', color='r', linewidth=lw)
ax[1,0].set_xlabel('$r_s$ [kpc]')
ax[1,0].set_ylim((0,11000))

# Release: Fig 5 (bottom middle) shows q_in; the alpha_in variant is kept commented out.
# bins = np.linspace(-3.4,-1.0,30)
# ax[1,1].hist(emulated_data_dreams[:,26], bins=bins, histtype='step', color='k', linewidth=lw)
# ax[1,1].hist(emulated_data_gse[:,26], bins=bins, histtype='step', color='r', linewidth=lw)
# ax[1,1].set_xlabel(r'$\alpha_\mathrm{in}$')
# ax[1,1].set_ylim((0,10000))

bins = np.linspace(0.2,1.0,30)
ax[1,1].hist(emulated_data_dreams[:,28], bins=bins, histtype='step', color='k', linewidth=lw)
ax[1,1].hist(emulated_data_gse[:,28], bins=bins, histtype='step', color='r', linewidth=lw)
ax[1,1].set_xlabel("q$_\mathrm{in}$")
ax[1,1].set_ylim((0,15000))

bins = np.linspace(0.2,1.0,30)
ax[1,2].hist(emulated_data_dreams[:,29], bins=bins, histtype='step', color='k', linewidth=lw)
ax[1,2].hist(emulated_data_gse[:,29], bins=bins, histtype='step', color='r', linewidth=lw)
ax[1,2].set_xlabel("q$_\mathrm{out}$")
ax[1,2].set_ylim((0,15000))

for i,axi in enumerate(ax.ravel()):
    axi.set_ylabel("Emulated Samples per Bin")
        

plt.show()
fig.savefig("plots/gse_comparison.pdf", bbox_inches='tight')

In [ ]:
fig, ax = set_plot_params(2,3,figsize=(24,12))
lw = 4
do_density = True


bins = np.linspace(-4.5,1.5,30)
ax[0,0].hist(dreams_data_all[:,9+5], bins=bins, histtype='step', color='k', label='DREAMS MW-Mass Galaxies', linewidth=lw, density=do_density)
ax[0,0].hist(dreams_data_gse[:,9+5], bins=bins, histtype='step', color='r', label='DREAMS GSE Analogs', linewidth=lw, density=do_density)
ax[0,0].hist(emulated_data_gse[:,9+5], bins=bins, histtype='step', color='b', label='Emulated GSE Analogs', linewidth=lw, density=do_density)
ax[0,0].set_xlabel('SFR [M$_\odot$ yr$^{-1}$]')
ax[0,0].legend(frameon=False, loc='upper right')
# ax[0,0].set_ylim((0,30000))

bins = np.linspace(1,10,30)
ax[0,1].hist(dreams_data_all[:,17+5], bins=bins, histtype='step', color='k', linewidth=lw, density=do_density)
ax[0,1].hist(dreams_data_gse[:,17+5], bins=bins, histtype='step', color='r', linewidth=lw, density=do_density)
ax[0,1].hist(emulated_data_gse[:,17+5], bins=bins, histtype='step', color='b', linewidth=lw, density=do_density)
ax[0,1].set_xlabel('$\mathrm{R_{disk}}$ [kpc]')
# ax[0,1].set_ylim((0,10000))

bins = np.linspace(8.4,10.2,30)
ax[0,2].hist(dreams_data_all[:,19+5], bins=bins, histtype='step', color='k', linewidth=lw, density=do_density)
ax[0,2].hist(dreams_data_gse[:,19+5], bins=bins, histtype='step', color='r', linewidth=lw, density=do_density)
ax[0,2].hist(emulated_data_gse[:,19+5], bins=bins, histtype='step', color='b', linewidth=lw, density=do_density)
ax[0,2].set_xlabel('$\mathrm{M_{*,halo}}$ [M$_\odot$]')
# ax[0,2].set_ylim((0,10000))

bins = np.linspace(11,65,30)
ax[1,0].hist(dreams_data_all[:,20+5], bins=bins, histtype='step', color='k', linewidth=lw, density=do_density)
ax[1,0].hist(dreams_data_gse[:,20+5], bins=bins, histtype='step', color='r', linewidth=lw, density=do_density)
ax[1,0].hist(emulated_data_gse[:,20+5], bins=bins, histtype='step', color='b', linewidth=lw, density=do_density)
ax[1,0].set_xlabel('$r_s$ [kpc]')
# ax[1,0].set_ylim((0,11000))

bins = np.linspace(-3.4,-1.0,30)
ax[1,1].hist(dreams_data_all[:,21+5], bins=bins, histtype='step', color='k', linewidth=lw, density=do_density)
ax[1,1].hist(dreams_data_gse[:,21+5], bins=bins, histtype='step', color='r', linewidth=lw, density=do_density)
ax[1,1].hist(emulated_data_gse[:,21+5], bins=bins, histtype='step', color='b', linewidth=lw, density=do_density)
ax[1,1].set_xlabel(r'$\alpha_\mathrm{in}$')
# ax[1,1].set_ylim((0,10000))

bins = np.linspace(0.2,1.0,30)
ax[1,2].hist(dreams_data_all[:,23+5], bins=bins, histtype='step', color='k', linewidth=lw, density=do_density)
ax[1,2].hist(dreams_data_gse[:,23+5], bins=bins, histtype='step', color='r', linewidth=lw, density=do_density)
ax[1,2].hist(emulated_data_gse[:,23+5], bins=bins, histtype='step', color='b', linewidth=lw, density=do_density)
ax[1,2].set_xlabel("q$_\mathrm{in}$")
# ax[1,2].set_ylim((0,15000))

for i,axi in enumerate(ax.ravel()):
    axi.set_ylabel("Emulated Samples per Bin")
        

plt.show()
# fig.savefig("plots/gse_direct_comparison.pdf", bbox_inches='tight')

In [ ]:
gse_props = {0:"tmm", 1:"tgse", 2:'Ratio', 4:'f', 5:'beta'}
mw_props = {9:'SFR', 17:'Rdisk', 19:'Mhalo', 20:'Rbreak', 21:'alpha_in', 23:'q_in'}
mw = 4
for prop in [0,1,2,4,5,9,17,19,20,21,23]:
    plt.figure()
    set_plot_params()
    plt.plot(dreams_data_all[:,prop], dreams_data_all[:,mw], 'k.')
    plt.plot(dreams_data_gse[:,prop], dreams_data_gse[:,mw], 'r.')
    # plt.ylabel(mw_props[mw])

    if prop < 6:
        plt.xlabel(gse_props[prop])
    else:
        plt.xlabel(mw_props[prop])
    
    plt.show()

In [ ]:
gse_props = {0:"tmm", 1:"tgse", 2:'Ratio', 4:'f', 5:'beta'}
mw_props = {9:'SFR', 17:'Rdisk', 19:'Mhalo', 20:'Rbreak', 21:'alpha_in', 23:'q_in'}

dreams_vector = []
gse_vector = []
emulator_vector = []
# Release: 13 dimensions as in the paper (M*,halo = 19 was missing)
for new_prop in [-3,-2,0,1,2,4,5,9,17,19,20,21,23]:

    prop = new_prop + 5
    
    all_data = dreams_data_all[:,prop] 
    data_mu = np.mean(all_data)
    data_std = np.std(all_data)
    norm_data = (all_data - data_mu) / data_std

    gse_data = dreams_data_gse[:,prop]
    norm_gse = (gse_data - data_mu) / data_std

    emulator_data = emulated_data_gse[:,prop]
    norm_emulator = (emulator_data - data_mu) / data_std

    dreams_vector.append(norm_data)
    gse_vector.append(norm_gse)
    emulator_vector.append(norm_emulator)

    # plt.figure()
    # bins = np.linspace(-3,5,20)
    # plt.hist(norm_data, bins=bins, density=True, histtype='step')
    # plt.hist(norm_emulator, bins=bins, density=True, histtype='step')
    # plt.show()

dreams_vector = np.array(dreams_vector).T
gse_vector = np.array(gse_vector).T
emulator_vector = np.array(emulator_vector).T

all_r = []
for sim in dreams_vector:
    r = np.sqrt(np.sum(np.square(dreams_vector - sim), axis=1))
    all_r.append(np.min(r[r!=0]))

gse_r = []
for gse in gse_vector:
    r = np.sqrt(np.sum(np.square(gse_vector - gse), axis=1))
    gse_r.append(np.min(r[r!=0]))

emulator_r = []
for sample in emulator_vector:
    r = np.sqrt(np.sum(np.square(gse_vector - sample), axis=1))
    emulator_r.append(np.min(r[r!=0]))
    
# plt.figure()
# set_plot_params()
# bins = np.linspace(0,4,20)
# plt.hist(all_r, bins=bins, density=True, histtype='step')
# plt.hist(gse_r, bins=bins, density=True, histtype='step')
# plt.hist(emulator_r, bins=bins, density=True, histtype='step')
# plt.show()

all_sorted = np.sort(all_r)
gse_sorted = np.sort(gse_r)
emulator_sorted = np.sort(emulator_r)


fig, ax = set_plot_params()
plt.plot(all_sorted, np.arange(len(all_sorted))/(len(all_sorted)-1), color='olivedrab', label='MW-Mass (Simulation)')
plt.plot(gse_sorted, np.arange(len(gse_sorted))/(len(gse_sorted)-1), color='gold', label='GSE-Analog (Simulation)')
plt.plot(emulator_sorted, np.arange(len(emulator_sorted))/(len(emulator_sorted)-1), color='red', label='GSE-Analog (Emulation)')
plt.xlabel("Normalized Distance")
plt.ylabel("CDF")
plt.legend(frameon=False)
plt.show()
fig.savefig("plots/gse_cdf.pdf", bbox_inches='tight')

In [ ]:
# Release: split by normalized distance as in the paper (was: the 15000 most distant samples)
dist_cut = 2.7
extrapolation = np.where(np.array(emulator_r) > dist_cut)[0]
interpolation = np.where(np.array(emulator_r) <= dist_cut)[0]

plt.figure()
set_plot_params()
plt.plot(all_sorted, np.arange(len(all_sorted))/(len(all_sorted)-1))
plt.plot(gse_sorted, np.arange(len(gse_sorted))/(len(gse_sorted)-1))
plt.plot(np.array(emulator_r)[extrapolation], np.arange(len(extrapolation))/(len(extrapolation)-1))
plt.plot(np.array(emulator_r)[interpolation], np.arange(len(interpolation))/(len(interpolation)-1))
plt.xlabel("Normalized Distance")
plt.ylabel("CDF")
plt.show()

fig, ax = set_plot_params(2,3,figsize=(24,12))
lw = 4
do_density = False


bins = np.linspace(-4.5,1.5,30)
ax[0,0].hist(emulated_data_dreams[:,9+5], bins=bins, histtype='step', color='k', label='Emulated MW-Mass Galaxies', linewidth=lw, density=do_density, alpha=0.5)
ax[0,0].hist(emulated_data_gse[:,9+5], bins=bins, histtype='step', color='r', label='Emulated GSE Analogs', linewidth=lw, density=do_density)
ax[0,0].hist(emulated_data_gse[interpolation,9+5], bins=bins, histtype='step', color='b', label='Interpolated GSE Analogs', linewidth=lw, density=do_density)
ax[0,0].hist(emulated_data_gse[extrapolation,9+5], bins=bins, histtype='step', color='c', label='Extrapolated GSE Analogs', linewidth=lw, density=do_density)
ax[0,0].set_xlabel('log SFR [M$_\odot$ yr$^{-1}$]')
ax[0,0].legend(frameon=False, loc='upper right', fontsize=18)
ax[0,0].set_ylim((0,35000))

bins = np.linspace(1,10,30)
ax[0,1].hist(emulated_data_dreams[:,17+5], bins=bins, histtype='step', color='k', linewidth=lw, density=do_density, alpha=0.5)
ax[0,1].hist(emulated_data_gse[:,17+5], bins=bins, histtype='step', color='r', linewidth=lw, density=do_density)
ax[0,1].hist(emulated_data_gse[interpolation,17+5], bins=bins, histtype='step', color='b', linewidth=lw, density=do_density)
ax[0,1].hist(emulated_data_gse[extrapolation,17+5], bins=bins, histtype='step', color='c', linewidth=lw, density=do_density)
ax[0,1].set_xlabel('$\mathrm{R_{disk}}$ [kpc]')
# ax[0,1].set_ylim((0,10000))

bins = np.linspace(8.4,10.2,30)
ax[0,2].hist(emulated_data_dreams[:,19+5], bins=bins, histtype='step', color='k', linewidth=lw, density=do_density, alpha=0.5)
ax[0,2].hist(emulated_data_gse[:,19+5], bins=bins, histtype='step', color='r', linewidth=lw, density=do_density)
ax[0,2].hist(emulated_data_gse[interpolation,19+5], bins=bins, histtype='step', color='b', linewidth=lw, density=do_density)
ax[0,2].hist(emulated_data_gse[extrapolation,19+5], bins=bins, histtype='step', color='c', linewidth=lw, density=do_density)
ax[0,2].set_xlabel('log $\mathrm{M_{*,halo}}$ [M$_\odot$]')
# ax[0,2].set_ylim((0,10000))

bins = np.linspace(11,65,30)
ax[1,0].hist(emulated_data_dreams[:,20+5], bins=bins, histtype='step', color='k', linewidth=lw, density=do_density, alpha=0.5)
ax[1,0].hist(emulated_data_gse[:,20+5], bins=bins, histtype='step', color='r', linewidth=lw, density=do_density)
ax[1,0].hist(emulated_data_gse[interpolation,20+5], bins=bins, histtype='step', color='b', linewidth=lw, density=do_density)
ax[1,0].hist(emulated_data_gse[extrapolation,20+5], bins=bins, histtype='step', color='c', linewidth=lw, density=do_density)
ax[1,0].set_xlabel('$r_s$ [kpc]')
# ax[1,0].set_ylim((0,11000))

bins = np.linspace(-3.4,-1.0,30)
ax[1,1].hist(emulated_data_dreams[:,21+5], bins=bins, histtype='step', color='k', linewidth=lw, density=do_density, alpha=0.5)
ax[1,1].hist(emulated_data_gse[:,21+5], bins=bins, histtype='step', color='r', linewidth=lw, density=do_density)
ax[1,1].hist(emulated_data_gse[interpolation,21+5], bins=bins, histtype='step', color='b', linewidth=lw, density=do_density)
ax[1,1].hist(emulated_data_gse[extrapolation,21+5], bins=bins, histtype='step', color='c', linewidth=lw, density=do_density)
ax[1,1].set_xlabel(r'$\alpha_\mathrm{in}$')
# ax[1,1].set_ylim((0,10000))

bins = np.linspace(0.2,1.0,30)
ax[1,2].hist(emulated_data_dreams[:,23+5], bins=bins, histtype='step', color='k', linewidth=lw, density=do_density, alpha=0.5)
ax[1,2].hist(emulated_data_gse[:,23+5], bins=bins, histtype='step', color='r', linewidth=lw, density=do_density)
ax[1,2].hist(emulated_data_gse[interpolation,23+5], bins=bins, histtype='step', color='b', linewidth=lw, density=do_density)
ax[1,2].hist(emulated_data_gse[extrapolation,23+5], bins=bins, histtype='step', color='c', linewidth=lw, density=do_density)
ax[1,2].set_xlabel("q$_\mathrm{in}$")
# ax[1,2].set_ylim((0,15000))

for i,axi in enumerate(ax.ravel()):
    axi.set_ylabel("Emulated Samples per Bin")
        

plt.show()
fig.savefig("plots/interpolation_vs_extrapolation.pdf", bbox_inches='tight')

In [ ]:
emulated_data_gse[0]

In [ ]:
fig, ax = set_plot_params(5,4,figsize=(37,37))
lw = 4

labels = [r'$\Omega_{\rm m}$', r'$\sigma_8$', 'log $\widebar{e}_w$', r'log $\kappa_w$', r'log $\epsilon_{f,\mathrm{high}}$']
labels += [r'$t_{\rm mm}$', r'$t_{\rm gse}$', r'$M_{\rm gse} / M_{\rm MW}$', r'N$_{\rm gse}$', '$f$', r'$\beta$']
labels += ['log $\mathrm{M_*}$ [M$_\odot$]', 'log $\mathrm{M_{*,sat}}$ [M$_\odot$]', 'log $\mathrm{M_{bary}}$ [M$_\odot$]', 'log SFR [M$_\odot$ yr$^{-1}$]']
labels += ['$\mathrm{R_{eff}}$ [kpc]', 'log $\mathrm{M_{TOT,bulge}}$ [M$_\odot$]', 'log $\mathrm{M_{*,bulge}}$ [M$_\odot$]', 'log $\mathrm{M_{BH}}$ [M$_\odot$]', '$\sigma_\mathrm{*,bulge}$ [km s$^{-1}$]']
labels += ['V$_\mathrm{circ}$  [km s$^{-1}$]', 'log $\mathrm{M_{*,disk}}$  [M$_\odot$]', '$\mathrm{R_{disk}}$ [kpc]', '$z_\mathrm{disk}$ [kpc]']
labels += ['log $\mathrm{M_{*,halo}}$ [M$_\odot$]', '$r_s$ [kpc]', r'$\alpha_\mathrm{in}$', r'$\alpha_\mathrm{out}$', "q$_\mathrm{in}$", "q$_\mathrm{out}$"]

plot_range = [(0.27, 0.36), (0.78, 0.888), (-0.045, 1.16), (0.568, 1.18), (-1.6, -0.4)]
plot_range += [(0,1),(0,1),(0,49),(0,3.2),(0,1),(-1,1)]
plot_range += [(9.7,11.2),(7,11),(10.6,11.4),(-4.5,1.5),(0.2,1),(9.4,10.8),(8.8,10.7),(6.5,9),(65,350),(100,300),(8.6,11.2),(1,8),(0,1.6),(8.4,10.2),(11,60),(-3.4,-1.2),(-5.5,-2.5),(0.4,0.95),(0.25,0.95)] 

cnt = 0
offset = 11
for i in range(19):

    if i == 12:
        continue
    
    mi, ma = plot_range[i+offset]
    bins = np.linspace(mi,ma,30)

    # if i in [11,14]:
    #     offset += 1
    
    row = cnt // 4
    col = cnt % 4

    if row == 2 and col == 0:
        print(np.mean(emulated_data_dreams[:,i+offset]), np.mean(emulated_data_tng[:,i+offset]))

    if cnt != -3:
        ax[row,col].hist(emulated_data_dreams[:,i+offset], bins=bins, histtype='step', color='k', label='Emulated MW-Mass Galaxies', linewidth=lw)
        ax[row,col].hist(emulated_data_gse[:,i+offset], bins=bins, histtype='step', color='r', label='Emulated GSE Analogs', linewidth=lw)
        # ax[row,col].hist(emulated_data_tng[:,i+offset], bins=bins, histtype='step', color='b', label='Emulated GSE Analogs', linewidth=lw)
        # ax[row,col].hist(emulated_data_tng_gse[:,i+offset], bins=bins, histtype='step', color='purple', label='Emulated GSE Analogs', linewidth=lw)
    else:
        ax[row,col].hist(np.power(10,emulated_data_dreams[:,i+offset]), bins=bins, histtype='step', color='k', label='Emulated MW-Mass Galaxies', linewidth=lw)
        ax[row,col].hist(np.power(10,emulated_data_gse[:,i+offset]), bins=bins, histtype='step', color='r', label='Emulated GSE Analogs', linewidth=lw)
        # ax[row,col].hist(np.power(10,emulated_data_tng[:,i+offset]), bins=bins, histtype='step', color='b', label='Emulated GSE Analogs', linewidth=lw)
        # ax[row,col].hist(np.power(10,emulated_data_tng_gse[:,i+offset]), bins=bins, histtype='step', color='purple', label='Emulated GSE Analogs', linewidth=lw)
    ax[row,col].set_xlabel(labels[i+offset], fontsize=28)
    ax[row,col].tick_params(labelsize=22)

    if cnt == 0:
        ax[row,col].legend(frameon=False, loc='upper right', fontsize=22)
        ax[row,col].set_ylim((0,12000))

    # if cnt == 3:
    #     ax[row,col].set_yscale("log")
    
    cnt += 1

for i,axi in enumerate(ax.ravel()):
    axi.set_ylabel("Emulated Samples per Bin", fontsize=28)

# ax[-1,1].axis("off")
ax[-1,2].axis("off")
ax[-1,3].axis("off")

plt.show()
fig.savefig("plots/1D_GSE_Comp.pdf", bbox_inches='tight')

In [ ]:
def normalize_data(data, obs_mean, obs_high, obs_low, weights=None, mu=None, mi=None, ma=None):

    if weights is not None:
        repeats = weights*1000
        wdata = np.repeat(data, repeats.astype(int))
    else:
        wdata = data

    high = np.quantile(wdata,0.999)
    low = np.quantile(wdata,0.001)
    lcut = wdata > low
    hcut = wdata < high
    cut = lcut & hcut
    wdata = wdata[cut]

    if mu is None:
        #remove sfr=0 galaxies
        if obs_mean == 0.2355:
            mu = np.mean(wdata[wdata>-3.8])
        else:
            mu = np.mean(wdata)

    if mi is None:
        mi = np.min(wdata)
    if ma is None:
        ma = np.max(wdata)

    data = np.array(list(wdata) + [obs_low,obs_mean,obs_high])

    low_cut = data < mu
    data[low_cut] = (mu - data[low_cut]) / (mi - mu)
    data[~low_cut] = (data[~low_cut] - mu) / (ma - mu)

    return data[:-3], data[-3:], mu, mi, ma

In [ ]:
# observed_mean  = [10.699, 9.431, 10.9294, -1.155, .2355, 0.70, 10.265, 10.190, 6.623, .175, 113, 238, 10.613, 2.51, 0.387, 8.740, 25, -2.5, -4.35, 0.65, 0.8]
# observed_high  = [10.778, 9.544, 10.9918, -1.093, .2865, 0.77, 10.281, 10.230, 6.643, .100, 116, 253, 10.711, 2.97, 0.457, 8.845, 35, -2.2, -3.70, 0.70, 0.9]
# observed_low   = [10.602, 9.279, 10.8670, -1.217, .1845, 0.63, 10.248, 10.146, 6.602, .250, 110, 223, 10.485, 2.06, 0.319, 8.602, 15, -2.8, -5.00, 0.60, 0.7]
observed_mean  = [10.699, 9.431, 10.9294, -1.155, .2355, 0.70, 10.265, 10.190, 6.623, .175, 113, 238, 10.613, 2.51, 8.740, 25, -2.5, -4.35, 0.65, 0.8] # no z disk
observed_high  = [10.778, 9.544, 10.9918, -1.093, .2865, 0.77, 10.281, 10.230, 6.643, .100, 116, 253, 10.711, 2.97, 8.845, 35, -2.2, -3.70, 0.70, 0.9]
observed_low   = [10.602, 9.279, 10.8670, -1.217, .1845, 0.63, 10.248, 10.146, 6.602, .250, 110, 223, 10.485, 2.06, 8.602, 15, -2.8, -5.00, 0.60, 0.7]

In [ ]:
galaxy_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_galaxy_dylan_cond.npy')
bulge_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_bulge_dylan_cond.npy')
disk_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_disk_dylan_cond.npy')
halo_data = np.load(NEHOD_TORCH + '/example/generated_samples/1M_CDM_SB_gse_prob_halo_dylan_cond.npy')

In [ ]:
binned_weights = np.load("binned_weights.npy")

In [ ]:
labels = ['$\mathrm{M_*}$', '$\mathrm{M_{*,sat}}$', '$\mathrm{M_{bary}}$', '$\mathrm{M_{bary}}$ / $\mathrm{M_{TOT}}$', 'SFR']
labels += ['$\mathrm{R_{eff}}$', '$\mathrm{M_{TOT,bulge}}$', '$\mathrm{M_{*,bulge}}$', '$\mathrm{M_{BH}}$', '$\mathrm{M_{*,bulge}}$/$\mathrm{M_{TOT,bulge}}$', '$\sigma_\mathrm{*,bulge}$']
labels += ['V$_\mathrm{circ}$', '$\mathrm{M_{*,disk}}$', '$\mathrm{R_{disk}}$'] #, '$z_\mathrm{disk}$']
labels += ['$\mathrm{M_{*,halo}}$', '$r_s$', r'$\alpha_\mathrm{in}$', r'$\alpha_\mathrm{out}$', "q$_\mathrm{in}$", "q$_\mathrm{out}$"]

In [ ]:
full_data = []
names = ['halo_tot_smass','halo_sat_smass', 'halo_bmass', 'halo_bmass_frac', 'halo_sfr']
ngal = len(names)
names += ['bulge_reff', 'bulge_mass', 'bulge_smass', 'bulge_bhmass', 'bulge_dm_frac', 'bulge_vdisp']
nbulge = len(names) - ngal
names += ['disk_vcirc', 'disk_smass', 'disk_size'] #, 'disk_height']
ndisk = len(names) - nbulge - ngal
names += ['halo_central_smass',  'halo_break', 'halo_slope_in', 'halo_slope_out', 'halo_q_in', 'halo_q_out']
nhalo = len(names) - nbulge - ndisk - ngal

In [ ]:
params = np.loadtxt(DREAMS_DATA + "/Parameters/CDM/MW_zooms/CDM_TNG_MW_SB5.txt", skiprows=1)

dreams_norm = []
dreams_obs = []
dreams_mu = []
dreams_mi = []
dreams_ma = []
for i,name in enumerate(names):
    this_data = np.load(f'violin_data/{name}.npy')
    boxes = np.load(f'violin_data/weights/{name}.npy').astype(int)
    nan_cut = np.isnan(this_data) | np.isnan(boxes) | np.isinf(this_data)
    good_data = this_data[~nan_cut]
    boxes = boxes[~nan_cut]

    weights = []
    for k in boxes:
        idx = get_bin_idx(params[k,2:5])
        w = binned_weights[idx[0]][idx[1]][idx[2]]
        weights.append(w)
    weights = np.array(weights)

    print(f"{name}, {np.min(good_data):.2f}, {np.mean(good_data):.2f}, {np.max(good_data):.2f}, {len(good_data)}")
    norm_data, obs_data, mu, mi, ma = normalize_data(good_data, observed_mean[i], observed_high[i], observed_low[i], weights)
    
    dreams_norm.append(norm_data)
    dreams_obs.append(obs_data)
    dreams_mu.append(mu)
    dreams_mi.append(mi)
    dreams_ma.append(ma)

In [ ]:
# dreams_mi[4] = -4.5

In [ ]:
fig, ax = set_plot_params(figsize=(15,7))
colors = ['k']*ngal + ['b']*nbulge + ['orange']*ndisk + ['m']*nhalo
i = 0
left_data = []
right_data = []
for all_data in [galaxy_data, bulge_data, disk_data, halo_data]:

    gse_cut = np.zeros(len(all_data),dtype=bool)
    end = 0
    while np.sum(gse_cut) < 1024:
        end += 100
        tcut = all_data[:end,7] < 0.5
        gse_tcut = (all_data[:end,8] < 0.5) & (all_data[:end,8] > 0.33)
        mcut = all_data[:end,9] < 10
        fcut = all_data[:end,11] > 0.5
        bcut = all_data[:end,12] > 0.5
        gse_cut = tcut & gse_tcut & mcut & fcut & bcut
    print(len(gse_cut), np.sum(gse_cut))

    weights = []
    for k in range(end):
        idx = get_bin_idx(all_data[k,2:5])
        w = binned_weights[idx[0]][idx[1]][idx[2]]
        weights.append(w)
    weights = np.array(weights)
    
    for data in all_data.T[13:]:
        for j in range(2):
            if j == 0:
                # continue
                # norm_data, obs_data, mi, ma = normalize_data(data[:end], observed_mean[i], observed_high[i], observed_low[i], weights=weights)
                norm_data = dreams_norm[i]
                obs_data = dreams_obs[i]
                left_data.append(norm_data)

                if obs_data[2] > obs_data[0]:
                    high = obs_data[2]
                    low = obs_data[0]
                else:
                    high = obs_data[0]
                    low = obs_data[2]
                
                frac = np.sum((norm_data < high) & (norm_data > low)) / len(norm_data)
                
                if colors[i] != colors[i-1]:
                    offset = 0.3
                else:
                    offset = 0.3
                
                ax.text(i+1-offset, 1.0, f'{frac*100:.1f}%', fontsize=12, color=colors[i])
                
                ylow = obs_data[1] - low # obs_data[0]
                yhigh = high - obs_data[1] # obs_data[2] - obs_data[1]

                ax.errorbar(i+1,obs_data[1],yerr=[[ylow],[yhigh]], fmt='ko', capsize=7, linewidth=1, markersize=3)
            
            else:
                mu = dreams_mu[i]
                mi = dreams_mi[i]
                ma = dreams_ma[i]    
                norm_data, obs_data, mu, mi, ma= normalize_data(data[:end][gse_cut], observed_mean[i], observed_high[i], observed_low[i], mu=mu, mi=mi, ma=ma, weights=weights[gse_cut])
                    
                right_data.append(norm_data)

                if obs_data[2] > obs_data[0]:
                    high = obs_data[2]
                    low = obs_data[0]
                else:
                    high = obs_data[0]
                    low = obs_data[2]
                
                frac = np.sum((norm_data < high) & (norm_data > low)) / len(norm_data)
                ax.text(i+1-.3, -1.1, f'{frac*100:.1f}%', fontsize=12, color='green')
        i += 1

left_violin = ax.violinplot(left_data, showextrema=False,side='low')
right_violin = ax.violinplot(right_data, showextrema=False,side='high')

ngal = 5
nbulge = 6
ndisk = 3
nhalo = 6

colors = ['k']*ngal + ['b']*nbulge + ['orange']*ndisk + ['m']*nhalo
for i,v in enumerate(left_violin['bodies']):
    v.set_facecolor(colors[i])
for i,v in enumerate(right_violin['bodies']):
    v.set_facecolor('green')

ax.plot([ngal+.5]*2,[-5,5], 'k--', alpha=0.5)
ax.plot([nbulge+ngal+.5]*2,[-5,5], 'k--', alpha=0.5)
ax.plot([nbulge+ndisk+ngal+.5]*2,[-5,5], 'k--', alpha=0.5)

ax.set_xticks(np.arange(len(labels))+1,labels,rotation=45, ha='right', fontsize=24)
ax.set_ylim((-1.3,1.3))
ax.set_xlim((0.5,ngal+nbulge+ndisk+nhalo+0.5))
ax.set_ylabel("Normalized MW Properties", fontsize=32)

ax.set_yticks([-1,0,1],['Min','Mean','Max'], fontsize=24)
ax.text(1, 1.1, 'Galaxy', fontsize=18)
ax.text(ngal+1, 1.1, 'Bulge', fontsize=18)
ax.text(ngal+nbulge+1, 1.1, 'Disk', fontsize=18)
ax.text(ngal+nbulge+ndisk+1, 1.1, 'Halo', fontsize=18)

In [ ]:
import matplotlib
matplotlib.__version__

In [ ]:
# fig.savefig("plots/violin_gse.pdf", bbox_inches='tight')

In [ ]:
fig, ax = set_plot_params(figsize=(15,7))
i = 0
left_data = []
right_data = []
for all_data in [galaxy_data, bulge_data, disk_data, halo_data]:
    
    for data in all_data.T[13:]:
        # norm_data, obs_data, mi, ma = normalize_data(data[:end], observed_mean[i], observed_high[i], observed_low[i], weights=weights)
        norm_data = dreams_norm[i]
        obs_data = dreams_obs[i]
        left_data.append(norm_data)

        if obs_data[2] > obs_data[0]:
            high = obs_data[2]
            low = obs_data[0]
        else:
            high = obs_data[0]
            low = obs_data[2]
        
        frac = np.sum((norm_data < high) & (norm_data > low)) / len(norm_data)
        
        if colors[i] != colors[i-1]:
            offset = 0.3
        else:
            offset = 0.3

        ylow = obs_data[1] - low # obs_data[0]
        yhigh = high - obs_data[1] # obs_data[2] - obs_data[1]
        
        ax.text(i+1-offset, 1.0, f'{frac*100:.1f}%', fontsize=12, color=colors[i])
        ax.errorbar(i+1,obs_data[1],yerr=[[ylow],[yhigh]], fmt='ko', capsize=7, linewidth=1, markersize=3)
        i += 1

left_violin = ax.violinplot(left_data, showextrema=False)

ngal = 5
nbulge = 6
ndisk = 3
nhalo = 6

colors = ['k']*ngal + ['b']*nbulge + ['orange']*ndisk + ['m']*nhalo
for i,v in enumerate(left_violin['bodies']):
    v.set_facecolor(colors[i])

ax.plot([ngal+.5]*2,[-5,5], 'k--', alpha=0.5)
ax.plot([nbulge+ngal+.5]*2,[-5,5], 'k--', alpha=0.5)
ax.plot([nbulge+ndisk+ngal+.5]*2,[-5,5], 'k--', alpha=0.5)

ax.set_xticks(np.arange(len(labels))+1,labels,rotation=45, ha='right', fontsize=24)
ax.set_ylim((-1.3,1.3))
ax.set_xlim((0.5,ngal+nbulge+ndisk+nhalo+0.5))
ax.set_ylabel("Normalized MW Properties", fontsize=32)

ax.set_yticks([-1,0,1],['Min','Mean','Max'], fontsize=24)
ax.text(1, 1.1, 'Galaxy', fontsize=18)
ax.text(ngal+1, 1.1, 'Bulge', fontsize=18)
ax.text(ngal+nbulge+1, 1.1, 'Disk', fontsize=18)
ax.text(ngal+nbulge+ndisk+1, 1.1, 'Halo', fontsize=18)

In [ ]:
# fig.savefig("plots/violin.pdf", bbox_inches='tight')